# Multithreading with Numba

**Optional deep dive, about 20 minutes.** Numba can parallelize independent loop iterations with `prange`. We will cap its thread count so the exercise does not silently consume every core.

In [ ]:
import os
import time
import numpy as np
import numba
from numba import jit, prange

test_mode = os.environ.get("PYHPC_TEST_MODE") == "1"
n_threads = min(4, numba.get_num_threads())
numba.set_num_threads(n_threads)
print("Numba threads used:", numba.get_num_threads())

## Add many independent values

The midpoint rule estimates the integral of `4 / (1 + x**2)` from 0 to 1, which equals pi. Each loop step can calculate its value independently, and the values are added at the end.

In [ ]:
@jit
def estimate_pi_serial(n_steps):
    total = 0.0
    step = 1.0 / n_steps
    for index in range(n_steps):
        x = (index + 0.5) * step
        total += 4.0 / (1.0 + x * x)
    return total * step


@jit(parallel=True)
def estimate_pi_parallel(n_steps):
    total = 0.0
    step = 1.0 / n_steps
    for index in prange(n_steps):
        x = (index + 0.5) * step
        total += 4.0 / (1.0 + x * x)
    return total * step

In [ ]:
n_steps = 2_000_000 if test_mode else 50_000_000

# Call both functions once before timing.
serial_result = estimate_pi_serial(n_steps)
parallel_result = estimate_pi_parallel(n_steps)
np.testing.assert_allclose(serial_result, np.pi, rtol=1e-10)
np.testing.assert_allclose(parallel_result, serial_result, rtol=1e-12)

started = time.perf_counter()
estimate_pi_serial(n_steps)
serial_time = time.perf_counter() - started

started = time.perf_counter()
estimate_pi_parallel(n_steps)
parallel_time = time.perf_counter() - started

print(f"Serial:   {serial_time:.3f} s")
print(f"Parallel: {parallel_time:.3f} s")
print(f"Speedup:  {serial_time / parallel_time:.2f}x")

## Check which loops used several threads

Numba can report which loops used several threads. Read this report instead of assuming that `parallel=True` changed the loop.

In [ ]:
estimate_pi_parallel.parallel_diagnostics(level=1)

## Limit the number of threads

Dask can start several workers, and each Numba function can also start several threads. Together they may start more threads than the CPUs you requested. Choose one tool to use most of the CPUs and set a small limit for the others.

## Takeaway

Use `prange` only when loop steps are independent or can safely add into one result. Call the function once before timing, check which loops Numba ran in parallel, and limit threads to the CPUs you requested.